In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# --- Interactive Jupyter Notebook with True vs Reconnected Signal Comparison ---

@widgets.interact(
    f_sig=widgets.FloatSlider(value=40.0, min=20.0, max=100.0, step=5.0, description='Signal Freq (Hz):', style={'description_width': 'initial'}, layout=widgets.Layout(width='700px')),
    Fs=widgets.FloatSlider(value=120.0, min=60.0, max=300.0, step=10.0, description='Sampling $F_s$ (Hz):', style={'description_width': 'initial'}, layout=widgets.Layout(width='700px'))
)
def update_sinc_aliasing_plot(f_sig, Fs):
    # Nyquist limit for a pure cosine of frequency f_sig is 2 * f_sig
    nyquist_freq = 2 * f_sig
    has_aliasing = Fs < nyquist_freq
    
    print(f"Signal Frequency: {f_sig} Hz | Sampling Fs: {Fs} Hz | Nyquist Limit: {nyquist_freq} Hz")
    if has_aliasing:
        print("-> WARNING: Fs < 2*f_sig -> ALIASING OCCURS (Reconstructed signal will mismatch the original!)")
    else:
        print("-> PROPER SAMPLING -> Perfect Reconstruction.")
        
    Ts = 1.0 / Fs
    
    # Time axis setup
    t_start = -0.01
    t_end = 0.08
    t_cont = np.linspace(t_start, t_end, 2000)
    
    # 1. True continuous signal (e.g., pure cosine)
    x_true = np.cos(2 * np.pi * f_sig * t_cont)
    
    # 2. Sample instances
    n_values = np.arange(-2, 10)
    t_samples = n_values * Ts
    x_samples = np.cos(2 * np.pi * f_sig * t_samples)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.grid(True, linestyle=':', alpha=0.7)
    ax.set_xlim(t_start, t_end)
    ax.set_ylim(-1.6, 1.8)
    ax.axhline(0, color='black', linewidth=1)
    
    # Change background color slightly if aliasing occurs
    if has_aliasing:
        ax.set_facecolor('#fff5f5')
        ax.set_title(r'ALIASING EFFECT: Reconstructed Signal Mismatches Original Due to Undersampling ($F_s < 2f_{max}$)', fontsize=11, fontweight='bold', color='darkred')
    else:
        ax.set_facecolor('#f0fff0')
        ax.set_title(r'PERFECT RECONSTRUCTION: Sinc Interpolation Matches Original ($F_s \geq 2f_{max}$)', fontsize=11, fontweight='bold', color='darkgreen')
    
    ax.set_xlabel(r'Time $t$ (s)', fontsize=11)
    ax.set_ylabel(r'Amplitude', fontsize=11)
    
    # Plot true continuous signal in solid blue for comparison
    ax.plot(t_cont, x_true, color='blue', linewidth=2, alpha=0.5, label='Original True Signal $x(t)$')
    
    # Total reconstructed signal accumulator via sinc pulses
    x_reconstructed = np.zeros_like(t_cont)
    
    # Plot individual scaled sinc pulses for visible samples
    for n, t_n, x_n in zip(n_values, t_samples, x_samples):
        if t_start <= t_n <= t_end:
            sinc_pulse = x_n * np.sinc(Fs * (t_cont - t_n))
            x_reconstructed += sinc_pulse
            # Draw individual red sinc curves (lighter alpha)
            ax.plot(t_cont, sinc_pulse, color='red', alpha=0.3, linewidth=1)
            # Draw vertical stem lines
            ax.plot([t_n, t_n], [0, x_n], color='black', linewidth=0.8, alpha=0.5)

    # Plot sample dots
    for t_n, x_n in zip(t_samples, x_samples):
        if t_start <= t_n <= t_end:
            ax.plot(t_n, x_n, marker='o', markersize=6, color='black', zorder=5)

    # Plot final reconstructed signal as a dashed darkgreen curve
    ax.plot(t_cont, x_reconstructed, color='darkgreen', linestyle='--', linewidth=2.5, label='Reconstructed Signal (Sinc sum)')
    
    ax.legend(loc='upper right', fontsize=10)
    plt.show()

    # --- Educational Output Guide ---
    print("\n" + "="*95)
    print(" EDUCATIONAL GUIDE: COMPARING TRUE VS RECONSTRUCTED SIGNAL UNDER ALIASING")
    print("="*95)
    print(" - The solid blue curve is the original high-frequency signal we wish we could capture.")
    print(" - The dashed green curve is what the Whittaker-Shannon sinc interpolation actually reconstructs")
    print("   based solely on the black sample dots.")
    print(" - When Fs drops below the Nyquist limit (2 * f_sig), notice how the green curve completely")
    print("   mismatches the blue curve: this visual discrepancy is the textbook proof of aliasing distortion!")
    print(" - NOTE ON BOUNDARY DISCREPANCIES (TRUNCATION ARTIFACTS):")
    print("   In the theoretical Whittaker-Shannon formula, reconstruction requires summing an infinite number")
    print("   of sinc pulses from -inf to +inf. If we only sum a finite set of samples near the edges of the plot,")
    print("   the missing tails of the outer sinc functions cause the reconstructed green curve to artificially")
    print("   fade or mismatch at the margins, even when proper sampling (no aliasing) is satisfied. Expanding")
    print("   the sample range across a wider window resolves this boundary effect.")
    print("="*95)